# CitoSkeleton: fine-tune Ask on a free GPU
Runtime → Change runtime type → T4 GPU. Then run each cell. Everything is made-up data; see training/README.md.

In [ ]:
!git clone https://github.com/oshwetank/llm-citation-checker.git cito
%cd cito
!pip -q install -r training/requirements.txt

In [ ]:
# Node for the data generator
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null && apt-get -qq install -y nodejs >/dev/null
!node training/gen.mjs --out training/data --train 20000 --test 1000 --seed 7
!node training/eval.mjs --data training/data/test.jsonl --oracle --limit 200

In [ ]:
# 0.8B fits a T4; use Qwen/Qwen3.5-2B on an A100
!python training/train.py --data training/data/train.jsonl --base Qwen/Qwen3.5-0.8B --out runs/cito-0.8b --epochs 2

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python training/export.py --model runs/cito-0.8b/merged --out runs/cito-0.8b/onnx --fp16 --upload YOUR_NAME/citoskeleton-qwen3.5-0.8b-onnx

In the extension: About → Settings → Built-in AI → **Your trained model** → `YOUR_NAME/citoskeleton-qwen3.5-0.8b-onnx` → **Use my trained model**, then **Test this computer**.